In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

In [ ]:
!nvidia-smi

In [ ]:
!pip install transformers datasets diffusers
# diffusers - generate images

In [ ]:
import torch
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio


In [ ]:
x = pipeline("sentiment-analysis",device="cuda")
res = x("I have dropped a couple of pounds")
print(res)

# can pass model as param.
# ner , Named Entity Recognition
# zero-shot-classification , candidate_labels can be passed

In [ ]:
image_gen = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2",
    torch_dtype=torch.float16,
    USE_SAFETENSORS = True,
    variant="fp16"
).to("cuda")

text = "Halong bay as a gibili image with superman flying in the air"
image = image_gen(prompt=text).images[0]
image

In [ ]:
# pipeline("text-to-speech","microsoft/speecht5_tts")
# more params regarding type or voice etc needed.

In [ ]:
import os
from huggingface_hub import InferenceClient
from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')  #uncomment when running

client = InferenceClient(
    provider="cerebras"
)

completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

In [ ]:
!hf auth login

In [ ]:
from huggingface_hub import login
from google.colab import userdata
# login(userdata.get('HF_TOKEN'))  #uncomment when running

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="meta-llama/Llama-3.1-8B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")

tokens = tokenizer.encode("This is a text message to test tokenization of lambda")
print(tokens)

In [ ]:
tokenizer.decode(tokens) #retuns text , includes special tokens , for ex: <begining of text>


In [ ]:
tokenizer.decode(tokens) #retuns text , includes special tokens , for ex: <begining of text>
tokenizer.batch_decode(tokens) #token as list
# tokenizer.vocab() #list all tokens
# tokenizer.get_added_vocab() #returns specialized tokens


In [ ]:
for tk in tokens:
  print(f"{tk} = {tokenizer.decode(tk)}")

In [ ]:
tokenizer.get_added_vocab()

In [ ]:
#Instruct are for a particular purpose , like chat
tokenizer_instruct = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")


In [ ]:
messages = [
    {"role":"system" , "content":"You are a Funny Assistant"},
    {"role":"user" , "content":"Tell me something"}
]

prompt = tokenizer_instruct.apply_chat_template(messages , tokenize=False,add_generation_prompt=True)
print(prompt)

In [ ]:
#startcoder_2 tokenizer useful for tokenizing code(programs)

In [ ]:
!pip install -q requests torch bitsandbytes

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM,TextStreamer,BitsAndBytesConfig
import torch

# Quantization ??? - converting a big model config into 4 bit
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

model = "meta-llama/Llama-3.1-8B-Instruct"
messages = [
    {"role":"system" , "content" : "You are a funny AI Assistant"},
    {"role":"user" , "content" : "Tell a joke about WWE"}
            ]


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model)
# tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages,return_tensors='pt').to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(model,device_map="auto",quantization_config = quant_config)


In [ ]:
output= model.generate(inputs,max_new_tokens=80,streamer=streamer)
# print(output)
# print(tokenizer.decode(output[0]))

In [ ]:
del tokenizer,streamer,model,inputs,output
torch.cuda.empty_cache()